# Weighted ensembles: performance and diversity (Prompt #11, Steps 12-14, 55-57, 76-77)

Weights from the skill on earlier blocks only, shrunk toward equal weights; diversity-aware weights that discount redundant errors and give a duplicate cluster one weight. How stable the weights are across blocks, and what a useless model and a duplicated model receive.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.ensemble` and `xauusd_quant.research.ensemble_*` and read back from `results/ensemble_research/`. Generate those first:

```
xq ensemble-research --timeframe 5m --report   # the meta walk-forward of every pair
xq ensemble-freeze --timeframe 5m              # pre-registered rule -> ENSEMBLE_SPEC
xq ensemble-finalize --timeframe 5m            # constituents, streaming, benchmark
xq ensemble-final-test --ensemble-spec ENS_REVERSION_5M_H5_V001   # once; logged
```

Every method is scored on the evaluation blocks 2013-2021 and fitted on earlier out-of-sample blocks only (purged by horizon + embargo). Probabilities, expected values and uncertainty proxies only - no signal, threshold, size or PnL.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.ensemble.config import load_ensemble_config

cfg = load_ensemble_config(ROOT / "config" / "ensemble.yaml")

TIMEFRAME = "5m"                      # 5m | 15m - change and re-run
TARGET, HORIZON = "mean_reversion", 5  # any pair of config/ensemble.yaml

out = cfg.results_path / TIMEFRAME
pair_dir = out / TARGET / f"h{HORIZON}"
if not pair_dir.exists():
    print(f"No ensemble research for {TIMEFRAME} {TARGET} h{HORIZON}. Run:  "
          f"xq ensemble-research --timeframe {TIMEFRAME} --report")

def table(name, where=None):
    """A stored table of the pair (or of the timeframe with where='tf'); None if absent."""
    base = out / "tables" if where == "tf" else pair_dir
    p = base / f"{name}.parquet"
    return pl.read_parquet(p) if p.exists() else None

def plot(name, where=None):
    """Show a stored figure of the pair (or of the timeframe with where='tf')."""
    base = out / "plots" if where == "tf" else pair_dir / "plots"
    p = base / f"{name}.png"
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f"no figure {p.name}")

def summary():
    p = pair_dir / "summary.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


## Weighted methods against the simple average and the best individual

In [ ]:
table('series_summary').filter(pl.col('series').str.contains('weighted')).select('series', 'mean', 'se', 'worst_block', 'mean_ece', 'minus_simple_average', 'wins_vs_simple_average', 'minus_best_individual', 'wins_vs_best_individual')

## Weights by block, entropy and effective number of models (Steps 55-56)

In [ ]:
table('weights').filter(pl.col('scheme').is_in(['performance_weighted_s50', 'diversity_weighted_l50', 'stacking']))

In [ ]:
plot('weight_stability')

## Weight stability summary

In [ ]:
summary().get('weight_stability')

## Noise model and duplicate model (Steps 76-77)

In [ ]:
json.loads((pair_dir / 'controls.json').read_text(encoding='utf-8'))